# Task 1: Multi-Class Gravitational Lens Classification

Classify simulated strong lensing images into three categories based on what dark matter substructure is present:

| Label | Folder | Physical meaning |
|-------|--------|-----------------|
| 0 | `no` | No substructure, smooth lens |
| 1 | `sphere` | CDM subhalo substructure |
| 2 | `vort` | Vortex / WDM-like substructure |

Images are single-channel (grayscale), 150x150, pre-normalized to [0, 1].
30,000 training images (10,000 per class), 7,500 validation.

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# make utils importable whether the notebook runs from task1/ or repo root
_here = Path().resolve()
_root = _here.parent if _here.name == "task1" else _here
sys.path.insert(0, str(_root))

from utils.models import LensClassifierEffNet
from utils.losses import LabelSmoothingCrossEntropy

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}  |  AMP: {use_amp}")

## Dataset

Each `.npy` file is a (H, W) or (1, H, W) float32 array already in [0, 1].
Labels come from the subdirectory name; no manual mapping needed.

Augmentation (train only): random h/v flips, random 90-degree rotations, and random erasing.
Random erasing blanks out a small rectangular patch set to the image mean, simulating PSF occlusion or detector artifacts.

In [ ]:
CLASS_NAMES = ["no", "sphere", "vort"]

class LensingDataset(Dataset):
    def __init__(self, root: str, split: str = "train", augment: bool = False):
        self.samples = []
        self.augment = augment
        split_dir = Path(root) / split
        for label, cls_name in enumerate(CLASS_NAMES):
            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                raise FileNotFoundError(f"Missing directory: {cls_dir}")
            for p in sorted(cls_dir.glob("*.npy")):
                self.samples.append((str(p), label))
        counts = np.bincount([lbl for _, lbl in self.samples], minlength=len(CLASS_NAMES))
        print(f"  {split}: {dict(zip(CLASS_NAMES, counts))}  total={len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 2:
            arr = arr[np.newaxis]
        img = torch.from_numpy(arr)
        if self.augment:
            img = self._augment(img)
        return img, label

    @staticmethod
    def _augment(img):
        if torch.rand(1) < 0.5:
            img = TF.hflip(img)
        if torch.rand(1) < 0.5:
            img = TF.vflip(img)
        k = int(torch.randint(0, 4, (1,)))
        if k:
            img = torch.rot90(img, k, dims=[-2, -1])
        if torch.rand(1) < 0.3:
            c, h, w = img.shape
            sh = int(h * torch.empty(1).uniform_(0.02, 0.15))
            sw = int(w * torch.empty(1).uniform_(0.02, 0.15))
            r0 = int(torch.randint(0, h - sh + 1, (1,)))
            c0 = int(torch.randint(0, w - sw + 1, (1,)))
            img = img.clone()
            img[:, r0:r0 + sh, c0:c0 + sw] = img.mean()
        return img


def get_sample_weights(dataset):
    counts = np.bincount([lbl for _, lbl in dataset.samples], minlength=len(CLASS_NAMES))
    class_w = 1.0 / (counts.astype(np.float64) + 1e-8)
    return torch.tensor([class_w[lbl] for _, lbl in dataset.samples])

## Mixup

Mixup blends pairs of images and their labels by a random ratio sampled from a Beta distribution.
It pushes the model toward smoother decision boundaries and acts as a strong regularizer on balanced datasets.

In [ ]:
def mixup_batch(imgs, labels, alpha, criterion, model):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    logits = model(mixed)
    loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[idx])
    return logits, loss

## Model

EfficientNet-B3 pretrained on ImageNet. Two things required to make it work well on grayscale astronomical images:

**Channel adaptation.** The first conv layer is replaced with a 1-channel version initialized by averaging the pretrained RGB weights. This preserves the scale of activations rather than starting from random weights.

**Internal normalization.** ImageNet statistics (mean=0.449, std=0.226 for grayscale) are baked into the model as a registered buffer. The data pipeline can pass raw [0, 1] tensors without any extra transforms.

**Differential learning rates.** Pretrained backbone gets lr=5e-5; the new classification head gets lr=3e-4. A uniform learning rate either leaves the backbone mostly unchanged or damages it.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DATA_DIR   = "../task1_data"   # adjust if running from a different working directory
EPOCHS     = 50
BATCH_SIZE = 64
LR_BACKBONE = 5e-5
LR_HEAD     = 3e-4
WEIGHT_DECAY = 1e-4
SMOOTHING    = 0.05
MIXUP_ALPHA  = 0.2
NUM_WORKERS  = 4
SAVE_DIR     = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Loading datasets...")
train_ds = LensingDataset(DATA_DIR, split="train", augment=True)
val_ds   = LensingDataset(DATA_DIR, split="val",   augment=False)
try:
    test_ds = LensingDataset(DATA_DIR, split="test", augment=False)
except FileNotFoundError:
    print("  No test split found; using val for final evaluation")
    test_ds = val_ds

sample_tensor, _ = train_ds[0]
in_channels = sample_tensor.shape[0]
print(f"  Input shape: {tuple(sample_tensor.shape)}")

sample_weights = get_sample_weights(train_ds)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))

model = LensClassifierEffNet(num_classes=3, in_channels=in_channels, pretrained=True)
model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: EfficientNet-B3  |  Trainable params: {n_params:,}")

criterion   = LabelSmoothingCrossEntropy(smoothing=SMOOTHING)
param_groups = model.get_param_groups(lr_backbone=LR_BACKBONE, lr_head=LR_HEAD)
optimizer   = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler   = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_BACKBONE * 10, LR_HEAD * 10],
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=25.0,
    final_div_factor=1e4,
)
scaler = torch.amp.GradScaler("cuda") if use_amp else None

## Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, mixup_alpha=0.2, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.amp.autocast("cuda", enabled=(scaler is not None)):
            if mixup_alpha > 0:
                logits, loss = mixup_batch(imgs, labels, mixup_alpha, criterion, model)
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    tta_transforms = [
        lambda x: x,
        lambda x: TF.hflip(x),
        lambda x: TF.vflip(x),
        lambda x: torch.rot90(x, 1, dims=[-2, -1]),
        lambda x: torch.rot90(x, 2, dims=[-2, -1]),
        lambda x: torch.rot90(x, 3, dims=[-2, -1]),
    ] if tta else [lambda x: x]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if tta:
            probs_list = [torch.softmax(model(torch.stack([tfm(im) for im in imgs])), dim=1)
                          for tfm in tta_transforms]
            probs  = torch.stack(probs_list).mean(0)
            logits = torch.log(probs + 1e-8)
        else:
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (probs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_probs), np.concatenate(all_labels)


def compute_roc_auc(probs, labels, n_classes=3):
    y_bin = label_binarize(labels, classes=list(range(n_classes)))
    try:
        return roc_auc_score(y_bin, probs, multi_class="ovr", average="macro")
    except Exception:
        return float("nan")


def compute_per_class_auc(probs, labels, n_classes=3):
    y_bin = label_binarize(labels, classes=list(range(n_classes)))
    return {CLASS_NAMES[i]: float(roc_auc_score(y_bin[:, i], probs[:, i]))
            for i in range(n_classes)}

## Training Loop

Best checkpoint is saved whenever validation AUC improves.
Final test evaluation uses test-time augmentation (6 geometric variants averaged).

In [ ]:
best_val_auc = 0.0
history      = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        mixup_alpha=MIXUP_ALPHA, scaler=scaler
    )
    scheduler.step()

    val_loss, val_acc, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device, tta=False
    )
    val_auc = compute_roc_auc(val_probs, val_labels)
    elapsed = time.time() - t0

    cur_lrs = [pg["lr"] for pg in optimizer.param_groups]
    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"Val loss {val_loss:.4f} acc {val_acc:.4f} AUC {val_auc:.4f} | "
        f"LR {cur_lrs[0]:.2e}/{cur_lrs[-1]:.2e} | {elapsed:.1f}s"
    )
    history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "val_auc": float(val_auc)})

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  New best val AUC: {best_val_auc:.4f}")

# Load best checkpoint and evaluate on test set with TTA
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device))
_, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion, device, tta=True)
test_auc  = compute_roc_auc(test_probs, test_labels)
per_class = compute_per_class_auc(test_probs, test_labels)

print(f"\nTest accuracy: {test_acc:.4f}  |  Test ROC-AUC (macro): {test_auc:.4f}")
print("Per-class AUC:", {k: f"{v:.4f}" for k, v in per_class.items()})

results = {
    "test_accuracy": test_acc, "test_roc_auc": float(test_auc),
    "per_class_auc": per_class, "best_val_auc": best_val_auc,
    "history": history,
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {SAVE_DIR}/results.json")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

epochs_x = [h["epoch"]      for h in history]
tr_loss  = [h["train_loss"] for h in history]
val_loss = [h["val_loss"]   for h in history]
tr_acc   = [h["train_acc"]  for h in history]
val_acc  = [h["val_acc"]    for h in history]
val_auc  = [h["val_auc"]    for h in history]

axes[0].plot(epochs_x, tr_loss, label="train")
axes[0].plot(epochs_x, val_loss, label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, tr_acc, label="train")
axes[1].plot(epochs_x, val_acc, label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, val_auc, color="green")
axes[2].axhline(max(val_auc), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(val_auc):.4f}")
axes[2].set_title("Val ROC-AUC"); axes[2].set_xlabel("Epoch")
axes[2].set_ylim([0.5, 1.0]); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## ROC Curves (Test Set)

In [ ]:
y_bin = label_binarize(test_labels, classes=[0, 1, 2])
colors = ["steelblue", "tomato", "seagreen"]

fig, ax = plt.subplots(figsize=(6, 5))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], test_probs[:, i])
    auc_val = per_class[cls]
    ax.plot(fpr, tpr, color=color, label=f"{cls}  (AUC={auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curves (macro AUC={test_auc:.4f})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()